# Step 3 of 3 — Prediction · Iteration 4

Your iteration-3 prediction notebook, with the changes iteration 4 needs. Each change is marked `CHANGED` or `ITERATION 4`:
- **Time frames.** History ends 2026-09-30, the forecast runs 2026-10-01 → 2026-12-31 (92 days, equal to the model's output length). All of it comes from `column_roles.json`.
- **Per-series covariates.** Each series is forecast with the shared calendar plus its own two (already scaled) spend columns, built from `series_cache/shared_cov.pkl` and `series_cache/spend_paths.npy`. Nothing is scaled here.
- **Run consistency.** The notebook stops if `run_metadata.json` and `column_roles.json` disagree on covariate columns, scaling constants or dates, which happens when Step 1 is re-run without re-training.
- **Shape check.** The old hard-coded `HIST_SHARE` was for 1 Sep – 7 Dec and does not match the new window. It is now recomputed from the cached sales of every series, one row per past year, aligned on Navratri and Diwali as before.
- **Covariate check** (last cell) covers the new columns: binary flags, `POST_GST_CUT`, petrol, and the spend paths in the forecast window (including whether the plan looks carried forward).

The NegBin decoding, block-wise prediction and output checks are unchanged.

In [ ]:
import os
import json
import gc
import calendar
import collections.abc
from datetime import datetime

import numpy as np
import pandas as pd
import torch

from darts import TimeSeries
from darts.models import TFTModel
from darts.utils.likelihood_models import NegativeBinomialLikelihood

In [ ]:
# Model name comes from Step 2's run_metadata.json -- no hand-typing.
# To score an older model instead, overwrite MODEL_NAME here (it must use the same series_cache).
with open(os.path.join(os.getcwd(), "run_metadata.json")) as f:
    META = json.load(f)
MODEL_NAME = META["model_name"]
print("Model:", MODEL_NAME)

In [ ]:
#Loading the CACHE_DIR
BASE = os.getcwd()
CACHE_DIR       = os.path.join(os.getcwd(), "series_cache")


#Loading the ROLES Json : This helps to avoid hardcoding the variables
with open(os.path.join(BASE,"column_roles.json")) as f:
    ROLES = json.load(f)

time_col,group_col,target_col = ROLES["time_col"],ROLES["group_col"],ROLES["target_col"]

FREQ = ROLES["freq"]
static_covariates = ROLES["static_covariates"]
shared_covariates = ROLES["shared_covariates"]
series_covariates = ROLES["series_covariates"]
future_covariates = ROLES["future_covariates"]
INPUT_CHUNK_LENGTH = ROLES["input_chunk_length"]

TRAIN_START    = pd.Timestamp(ROLES["train_start"])
VAL_END        = pd.Timestamp(ROLES["val_end"])
FORECAST_START = pd.Timestamp(ROLES["forecast_start"])
FORECAST_END = pd.Timestamp(ROLES["forecast_end"])
HORIZON = (FORECAST_END - FORECAST_START).days + 1

# ITERATION 4: the model, the data and the roles file must all come from the same run.
assert META["future_covariate_columns"] == future_covariates, \
    "run_metadata.json and column_roles.json list different covariate columns -- re-run Steps 1 and 2 together."
assert META["scaling"] == ROLES["scaling"], \
    "run_metadata.json and column_roles.json have different scaling constants -- the model was trained on another Step 1 run."
assert META["dates"]["forecast_start"] == ROLES["forecast_start"] and \
       META["dates"]["forecast_end"] == ROLES["forecast_end"], "run_metadata.json and column_roles.json disagree"

safe_name = lambda k: str(k).replace("<>", "_").replace("/", "_").replace("\\", "_")

In [ ]:
# ---------------- SEQUENCES ----------------
class History(collections.abc.Sequence):
    """Series history to forecast from. Reads the cached 'val' split,
    which ends on the day before FORECAST_START."""

    def __init__(self, keys, statics):
        self.keys, self.statics = keys, statics

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, i):
        if isinstance(i, slice):
            return [self[j] for j in range(*i.indices(len(self)))]
        if i < 0:
            i += len(self)
        if not 0 <= i < len(self):
            raise IndexError(i)
        with np.load(os.path.join(CACHE_DIR, f"{safe_name(self.keys[i])}.npz")) as z:
            sales, start = z["val_sales"], str(z["val_start"])
        times = pd.date_range(start, periods=len(sales), freq=FREQ)
        if times[-1] != FORECAST_START - pd.Timedelta(days=1):
            raise ValueError(f"History for {self.keys[i]} ends on {times[-1]}.")
        if len(sales) < INPUT_CHUNK_LENGTH:
            raise ValueError(f"History for {self.keys[i]} has {len(sales)} days; the encoder needs {INPUT_CHUNK_LENGTH}.")
        return TimeSeries.from_times_and_values(
            times,
            sales.reshape(-1, 1).astype(np.float32),
            columns=[target_col],
            static_covariates=self.statics[i],
        )


# ITERATION 4: replaces SharedCov. Same class as in Step 2 (the globals it needs are loaded two cells below).
class SeriesCov(collections.abc.Sequence):
    """Future covariates, one TimeSeries per series: the shared calendar columns followed by
    that series' spend columns. Series that share a spend path share one TimeSeries object."""

    def __init__(self, path_ids, cache_in_ram=True):
        self.path_ids = np.asarray(path_ids, dtype=np.int64)
        assert ((self.path_ids >= 0) & (self.path_ids < len(SPEND_PATHS))).all(), "a series has no valid spend path"
        self._ram = {} if cache_in_ram else None

    def _build(self, pid):
        values = np.concatenate([CAL_VALUES, SPEND_PATHS[pid]], axis=1).astype(np.float32, copy=False)
        return TimeSeries.from_times_and_values(CAL_INDEX, values, columns=COV_COLUMNS)

    def __len__(self):
        return len(self.path_ids)

    def __getitem__(self, i):
        if isinstance(i, slice):
            return [self[j] for j in range(*i.indices(len(self)))]
        if i < 0:
            i += len(self)
        if not 0 <= i < len(self):
            raise IndexError(i)
        pid = int(self.path_ids[i])
        if self._ram is None:
            return self._build(pid)
        if pid not in self._ram:
            self._ram[pid] = self._build(pid)
        return self._ram[pid]

In [ ]:
print(f"Forecast: {FORECAST_START.date()} -> {FORECAST_END.date()} ({HORIZON} days)")

In [ ]:
BATCH_SIZE = 512        # raise until GPU memory complains
BLOCK      = 5000       # series per block, written to disk as it goes
LIMIT      = None       # set to e.g. 2000 for a quick smoke test

In [ ]:
# CHANGED for iteration 3: series list and static codes come from Step 2's series_index.parquet.
# The pickled OrdinalEncoder (static_cov_transformer.pkl) no longer exists -- codes are already
# integers from category_vocab.json, and the model looks them up in embedding tables.
from tft_checks import assert_embeddings

series_index = pd.read_parquet(os.path.join(CACHE_DIR, "series_index.parquet"))
sel = series_index[series_index["in_forecast"]]
if LIMIT:
    sel = sel.iloc[:LIMIT]

keys    = sel["key"].tolist()
statics = [pd.DataFrame([row], columns=static_covariates, dtype=np.float32)
           for row in sel[static_covariates].to_numpy()]
path_ids = sel["spend_path"].to_numpy()

print(f"Series to forecast: {len(keys):,}")
print(f"  ({int((~series_index['in_forecast']).sum()):,} series skipped: no sales at all)")

# ITERATION 4: the covariates are the shared calendar (pickled by Step 2) plus the spend paths (saved by Step 2).
SHARED_COV  = TimeSeries.from_pickle(os.path.join(CACHE_DIR, "shared_cov.pkl"))
SPEND_PATHS = np.load(os.path.join(CACHE_DIR, "spend_paths.npy"))
CAL_INDEX   = SHARED_COV.time_index
CAL_VALUES  = SHARED_COV.values(copy=False)
COV_COLUMNS = list(SHARED_COV.components) + list(series_covariates)

if SHARED_COV.end_time() < FORECAST_END:
    raise ValueError(f"Covariates end {SHARED_COV.end_time().date()}, need {FORECAST_END.date()}")
if SHARED_COV.start_time() > FORECAST_START - pd.Timedelta(days=INPUT_CHUNK_LENGTH):
    raise ValueError(f"Covariates start {SHARED_COV.start_time().date()}, but the encoder needs them from "
                     f"{(FORECAST_START - pd.Timedelta(days=INPUT_CHUNK_LENGTH)).date()}")
assert COV_COLUMNS == future_covariates, \
    f"Covariate columns/order differ from column_roles.json:\n  built : {COV_COLUMNS[:3]} ... {COV_COLUMNS[-3:]}\n  roles : {future_covariates[:3]} ... {future_covariates[-3:]}"
assert SPEND_PATHS.shape[1] == len(CAL_INDEX) and SPEND_PATHS.shape[2] == len(series_covariates), \
    f"spend_paths.npy has shape {SPEND_PATHS.shape}, which does not fit the calendar ({len(CAL_INDEX)} days)."
assert META["n_spend_paths"] == len(SPEND_PATHS), "spend_paths.npy is not the one the model was trained with -- re-run Step 2."
assert ((path_ids >= 0) & (path_ids < len(SPEND_PATHS))).all(), "Some series have no spend path."

_n_paths = len(np.unique(path_ids))
_est_gb = _n_paths * len(CAL_INDEX) * len(COV_COLUMNS) * 4 / 1e9
COV_IN_RAM = _est_gb <= 6.0
print(f"spend paths used: {_n_paths:,} of {len(SPEND_PATHS):,} | covariate RAM if kept: {_est_gb:.2f} GB -> "
      f"{'kept in RAM' if COV_IN_RAM else 'built on demand'}")
ALL_COV = SeriesCov(path_ids, cache_in_ram=COV_IN_RAM)

# Darts pickles the training callbacks inside the saved model, so a fresh kernel must be able to
# resolve the class name used in Step 2. This is an inert stand-in; it does nothing at predict time.
from pytorch_lightning import Callback
class TBDiagnostics(Callback):
    def __init__(self, *args, **kwargs):
        pass

# load BEST weights -- model.predict() on an in-memory model uses the LAST
# epoch, which with patience=5 is 5 epochs past what early stopping picked
model = TFTModel.load_from_checkpoint(MODEL_NAME, best=True)
print(f"Loaded best checkpoint: {MODEL_NAME}")

# the model being used for the forecast must have its embedding tables
_rep, _n = assert_embeddings(model.model, static_covariates,
                             {c: tuple(v) for c, v in META["embedding_sizes"].items()})
print(f"Embedding check passed: {len(_rep)} static columns, {_n:,} embedding parameters.")

# ITERATION 4: ... and the number of future inputs it was built with
try:
    _n_future = len(model.model.variables_meta["input"]["future_covariate"])
    assert _n_future == len(future_covariates) + 1, (
        f"The loaded model has {_n_future} future inputs; expected {len(future_covariates)} covariates + 1 relative index.")
    print(f"Covariate check passed: the model has {_n_future} future inputs ({len(future_covariates)} covariates + 1 relative index).")
except (AttributeError, KeyError) as e:
    print(f"Covariate-count check skipped ({type(e).__name__}: {e}).")

In [ ]:
# Every execution gets a fresh directory so old decoding results cannot be reused.
from uuid import uuid4
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid4().hex[:8]
OUT_DIR = os.path.join(
    BASE, "predictions_2026_negbin_torch_convention", RUN_ID
)
print("Output directory:", OUT_DIR)

In [ ]:
# ---------------- NEGATIVE-BINOMIAL DECODING ----------------
# Darts 0.40 returns (r, p) passed to torch.distributions.NegativeBinomial.
# PyTorch mean: r*p/(1-p). SciPy nbinom requires probability 1-p.
# Do not interpret Darts' internal variable name "mu" as the reported mean.

from scipy import stats
QUANTILES = [45,55,70,75]


def summarise(p_ts):
    """Decode the predictive distribution using PyTorch's probability convention."""
    expected = [f"{target_col}_r", f"{target_col}_p"]
    if list(p_ts.components) != expected:
        raise ValueError(f"Expected {expected}, received {list(p_ts.components)}")
    v = p_ts.values(copy=False).astype(np.float64)
    r, pr = v[:, 0], v[:, 1]
    if (not np.isfinite(v).all() or np.any(r <= 0)
            or np.any((pr <= 0) | (pr >= 1))):
        raise ValueError("Invalid negative-binomial parameters; inspect precision/output.")
    # Fail on invalid parameters rather than silently clipping them.
    out = {"PRED_MEAN": r * pr / (1.0 - pr)}
    for q_int in QUANTILES:
        out[f"PRED_Q{q_int}"] = stats.nbinom.ppf(q_int / 100.0, r, 1.0 - pr)
    if not all(np.isfinite(x).all() for x in out.values()):
        raise ValueError("Non-finite decoded predictions.")
    return out


# Check the installed likelihood implementation before forecasting all series.
# Synthetic raw outputs exercise both Darts' parameter export and its training
# distribution. This check is deterministic and does not run the TFT network.
import darts
print("Darts version:", darts.__version__)
if not isinstance(model.likelihood, NegativeBinomialLikelihood):
    raise TypeError("This notebook requires NegativeBinomialLikelihood.")
if not keys or len(set(keys)) != len(keys):
    raise ValueError("Forecast series keys must be non-empty and unique.")
if not 0 < HORIZON <= model.output_chunk_length:
    raise ValueError("Parameter prediction requires horizon <= output_chunk_length.")

_raw = torch.tensor([[[[-2.0, -1.0]], [[0.4, -0.2]], [[3.0, 1.5]]]],
                    dtype=torch.float64)
_lk = model.likelihood
_dist = _lk._distr_from_params(_lk._params_from_output(_raw))
_exported = _lk.predict_likelihood_parameters(_raw).detach().cpu().numpy()
_probe_ts = TimeSeries.from_times_and_values(
    pd.date_range("2000-01-01", periods=3, freq="D"),
    _exported.reshape(3, 2), columns=[f"{target_col}_r", f"{target_col}_p"],
)
_probe_summary = summarise(_probe_ts)
np.testing.assert_allclose(
    _probe_summary["PRED_MEAN"], _dist.mean.detach().cpu().numpy().ravel(),
    rtol=1e-10, atol=1e-10,
)
# Verify the SciPy probability conversion against Torch's probability mass.
_r, _p = _exported.reshape(3, 2).T
_counts = torch.tensor([0.0, 2.0, 5.0], dtype=torch.float64).reshape(1, 3, 1)
np.testing.assert_allclose(
    stats.nbinom.logpmf(_counts.numpy().ravel(), _r, 1.0 - _p),
    _dist.log_prob(_counts).detach().cpu().numpy().ravel(),
    rtol=1e-10, atol=1e-10,
)
print("PASS: decoded means and SciPy probabilities match the installed likelihood.")

# ---------------- FORECAST ----------------
EXPECTED_PARAMS = [f"{target_col}_r", f"{target_col}_p"]
EXPECTED_COLS   = {"PRED_MEAN", *(f"PRED_Q{q}" for q in QUANTILES)}

os.makedirs(OUT_DIR, exist_ok=True)
n_blocks = (len(keys) + BLOCK - 1) // BLOCK
paths, t0 = [], datetime.now()

for b in range(n_blocks):
    lo, hi = b * BLOCK, min((b + 1) * BLOCK, len(keys))
    path = os.path.join(OUT_DIR, f"{MODEL_NAME}_block_{b:04d}.parquet")
    paths.append(path)

    preds = model.predict(
        n=HORIZON,
        series=History(keys[lo:hi], statics[lo:hi]),
        future_covariates=ALL_COV[lo:hi],     # ITERATION 4: each series gets calendar + its own spend path
        predict_likelihood_parameters=True,   # 1 pass instead of num_samples passes
        num_samples=1,
        batch_size=BATCH_SIZE,
        verbose=False,
    )

    if len(preds) != hi - lo:
        raise ValueError("Prediction count does not match the requested series.")
    for p_ts in preds:
        if (len(p_ts) != HORIZON or p_ts.start_time() != FORECAST_START
                or p_ts.end_time() != FORECAST_END):
            raise ValueError("Forecast dates do not match the requested horizon.")

    pd.concat(
        [pd.DataFrame({group_col: k, time_col: p_ts.time_index, **summarise(p_ts)})
         for k, p_ts in zip(keys[lo:hi], preds)],
        ignore_index=True,
    ).to_parquet(path, index=False)

    el = (datetime.now() - t0).total_seconds()
    print(f"[{b+1}/{n_blocks}] {hi:,}/{len(keys):,} | {hi/el:,.0f} series/s | "
          f"ETA {(len(keys)-hi)/(hi/el)/60:.0f} min")

    del preds
    gc.collect()


# ---------------- OUTPUT ----------------
df = pd.concat([pd.read_parquet(p) for p in paths], ignore_index=True)
out = os.path.join(OUT_DIR, f"{MODEL_NAME}_predictions.parquet")
if len(df) != len(keys) * HORIZON or df.duplicated([group_col, time_col]).any():
    raise ValueError("Unexpected row count or duplicate series/date predictions.")
df.to_parquet(out, index=False)

print(f"\nWritten -> {out}")
print(f"  rows: {len(df):,} | elapsed: {(datetime.now()-t0).total_seconds()/60:.1f} min")
for c in ["PRED_MEAN"] + [f"PRED_Q{q}" for q in QUANTILES]:
    print(f"  {c}: {df[c].sum():,.0f} ({df[c].sum()/1e5:.2f} lacs)")

In [ ]:
# Read the predictions back from the path this run just wrote,
# rather than a hardcoded absolute path to the previous (wrong) output.
parquet_pred = pd.read_parquet(out)
print(f"{len(parquet_pred):,} rows from {out}")

In [ ]:
parquet_pred.head()

In [ ]:
parquet_pred[time_col] = pd.to_datetime(parquet_pred[time_col])
parquet_pred["MONTH_NAME"] = parquet_pred[time_col].dt.month_name()

In [ ]:
parquet_pred.groupby("MONTH_NAME",as_index=False).agg(TOTAL_PREDICTIONS=('PRED_Q70','sum')).to_csv(r"Iteration_4_agg_results_by_month.csv",index=False)

## Shape check — predicted month split vs. what the past festive seasons did
The total being plausible means little on its own; the failure mode here was always the **shape**.

**ITERATION 4:** the forecast window is now 1 Oct – 31 Dec 2026, so the old hard-coded `HIST_SHARE` (1 Sep – 7 Dec, from an 8% subset of series) no longer applies.
The cell below rebuilds it from `daily_total_sales.npy`, the sales of **every** series that Step 2 cached, with the same calendar-drift correction as before:
each past year is cut into October / November / December using its own Navratri (`N`) and Diwali (`D`) dates, offset by the same number of days as in 2026.
For 2026 that is: October = N−10 … D−8, November = D−7 … D+22, December = D+23 … D+53 (the first segment starts at the window start counted from `N`; every later boundary is counted from `D`).
Past years whose shifted window runs outside the cached history are skipped, and the cell prints the exact dates it used so you can check them.

In [ ]:
def window_shares_history():
    """Month shares of the forecast window in each past year, aligned on Navratri (N) and Diwali (D)."""
    daily = pd.Series(np.load(os.path.join(CACHE_DIR, "daily_total_sales.npy")),
                      index=pd.date_range(TRAIN_START, VAL_END, freq=FREQ))
    cal_ = pd.read_parquet(os.path.join(BASE, ROLES["calendar_path"]), columns=[time_col, "N", "D"])
    cal_[time_col] = pd.to_datetime(cal_[time_col])

    def anchors(year):
        c = cal_[(cal_[time_col].dt.year == year) & (cal_[time_col].dt.month >= 9)]
        n, d = c.loc[c["N"] != 0, time_col], c.loc[c["D"] != 0, time_col]
        if len(n) != 1 or len(d) != 1:
            return None
        return n.iloc[0], d.iloc[0]

    a26 = anchors(FORECAST_START.year)
    if a26 is None:
        raise ValueError(f"Could not find exactly one N and one D day between September and December {FORECAST_START.year}.")
    N26, D26 = a26
    print(f"{FORECAST_START.year}: N (Navratri) = {N26.date()}, D (Diwali) = {D26.date()}")

    segs = []        # (month name, start, end, anchor used for the start)
    for k, per in enumerate(pd.period_range(FORECAST_START, FORECAST_END, freq="M")):
        s = max(per.start_time.normalize(), FORECAST_START)
        e = min(per.end_time.normalize(), FORECAST_END)
        segs.append((calendar.month_name[per.month], s, e, "N" if k == 0 else "D"))

    shares = {}
    for year in range(TRAIN_START.year, FORECAST_START.year):
        a = anchors(year)
        if a is None:
            print(f"{year}: skipped (no single N / D day found)")
            continue
        Ny, Dy = a
        totals, spans, ok = {}, {}, True
        for name, s, e, s_anchor in segs:
            start = (Ny if s_anchor == "N" else Dy) + (s - (N26 if s_anchor == "N" else D26))
            end = Dy + (e - D26)
            if start < daily.index[0] or end > daily.index[-1]:
                ok = False
                break
            totals[name] = float(daily.loc[start:end].sum())
            spans[name] = f"{start.date()}..{end.date()}"
        if not ok or sum(totals.values()) <= 0:
            print(f"{year}: skipped (shifted window leaves the cached history)")
            continue
        tot = sum(totals.values())
        print(f"{year}: N = {Ny.date()}, D = {Dy.date()} | " + " | ".join(f"{n}: {spans[n]}" for n in totals))
        for name in totals:
            shares.setdefault(name, {})[str(year)] = round(100 * totals[name] / tot, 1)
    if not shares:
        raise ValueError("No past year could be aligned; cannot build the historical shape.")
    for name in shares:
        shares[name]["mean"] = round(float(np.mean(list(shares[name].values()))), 1)
    return shares


HIST_SHARE = window_shares_history()      # % of the forecast-window total, by month and past year

In [ ]:
# ---------------- SHAPE CHECK ----------------
parquet_pred[time_col]  = pd.to_datetime(parquet_pred[time_col])
parquet_pred["MONTH_NAME"] = parquet_pred[time_col].dt.month_name()

ORDER = list(HIST_SHARE)
COLS  = ["PRED_MEAN"] + [f"PRED_Q{q}" for q in QUANTILES]

split = (parquet_pred.groupby("MONTH_NAME")[COLS].sum()
         .reindex(ORDER))

print("HISTORICAL SHARE OF THE WINDOW (%), drift-corrected")
print(pd.DataFrame(HIST_SHARE).T.to_string())

print("\nTOTALS BY MONTH")
print(split.round(0).to_string())

print("\nSHARE OF WINDOW (%) -- predicted vs actual history")
share = 100 * split / split.sum().replace(0, np.nan)
share.insert(0, "HIST_mean", [HIST_SHARE[m]["mean"] for m in ORDER])
print(share.round(1).to_string())

gap = share["PRED_MEAN"] - share["HIST_mean"]
print("\nPRED_MEAN share minus historical mean share (pp):")
print(gap.round(1).to_string())
print("\nNote: the historical shares are not a forecast. A year whose festive peak was larger or earlier moves them, so read the gap as a prompt to look, not as an error.")

## Covariate check — what the model was given for the forecast window
The cell below reads the two artifacts inference uses (the pickled calendar and the spend paths) and prints what the forecast window looks like.
**Things to look for:**
- `FAIL` lines. They mean the model is being given missing or invalid numbers.
- `Days in the window with NO festive flag`: on those days the model can only extrapolate the baseline from the encoder.
- *spend paths whose whole forecast window equals the last history value*: a high share means the Oct–Dec plan is carried forward.
- *Petrol* and *spend* ranges next to the training range: values outside it are extrapolation.

In [ ]:
import os, json
import numpy as np
import pandas as pd
from darts import TimeSeries

BASE      = os.getcwd()
CACHE_DIR = os.path.join(os.getcwd(), "series_cache")

with open(os.path.join(BASE, "column_roles.json")) as f:
    ROLES = json.load(f)
FC_START = pd.Timestamp(ROLES["forecast_start"])
FC_END   = pd.Timestamp(ROLES["forecast_end"])
TR_END   = pd.Timestamp(ROLES["train_end"])
N_DAYS   = (FC_END - FC_START).days + 1

time_col = ROLES["time_col"]
binary_covariates = ROLES["binary_covariates"]
shared_covariates = ROLES["shared_covariates"]
series_covariates = ROLES["series_covariates"]
petrol_cols       = ROLES["shared_continuous_covariates"]

N_BLOCK = [f"N-{i}" for i in range(16, 0, -1)] + ["N"] + [f"N+{i}" for i in range(1, 11)]
D_BLOCK = [f"D-{i}" for i in range(3, 0, -1)] + ["D"] + [f"D+{i}" for i in range(1, 7)]


def frame_from_pickle(path):
    ts = TimeSeries.from_pickle(path)
    df = ts.to_dataframe().reset_index()
    df.columns = [time_col] + list(ts.components)
    return df


# ------------------------------------------------------------------ load both
sources = {}

pkl = os.path.join(CACHE_DIR, "shared_cov.pkl")
if os.path.exists(pkl):
    sources["shared_cov.pkl (what inference uses)"] = frame_from_pickle(pkl)
else:
    print("shared_cov.pkl NOT FOUND -- inference would have rebuilt from parquet.")

pq = os.path.join(BASE, "shared_calendar.parquet")
if os.path.exists(pq):
    d = pd.read_parquet(pq)
    d[time_col] = pd.to_datetime(d[time_col])
    sources["shared_calendar.parquet (source)"] = d


for label, cal in sources.items():
    print("=" * 70)
    print(label)
    print("=" * 70)

    cal[time_col] = pd.to_datetime(cal[time_col])
    print(f"Date range : {cal[time_col].min().date()} -> {cal[time_col].max().date()}")
    print(f"Columns    : {len(cal.columns) - 1}")

    if cal[time_col].max() < FC_END:
        print(f"\n*** FAIL: calendar ends before {FC_END.date()}. "
              f"Covariates are missing for part of the forecast. ***")

    win = cal[(cal[time_col] >= FC_START) & (cal[time_col] <= FC_END)]
    print(f"Rows in {FC_START.date()} - {FC_END.date()}: {len(win)} (expected {N_DAYS})")

    if len(win) == 0:
        print("\n*** FAIL: no calendar rows in the forecast window at all. ***\n")
        continue

    # --- NaN check: NaNs silently poison the whole forecast ------------------
    nan_cols = [c for c in win.columns if c != time_col and win[c].isna().any()]
    if nan_cols:
        print(f"\n*** FAIL: {len(nan_cols)} columns contain NaN in the window: "
              f"{nan_cols[:8]} ***")
    else:
        print("NaN check : clean")

    # --- ITERATION 4: binary flags really are 0/1 ------------------------------
    present_bin = [c for c in binary_covariates if c in cal.columns]
    bad_bin = [c for c in present_bin if not cal[c].isin([0, 1]).all()]
    missing_bin = [c for c in binary_covariates if c not in cal.columns]
    if missing_bin:
        print(f"*** FAIL: binary columns missing from this calendar: {missing_bin[:8]} ***")
    print("Binary check: " + ("all 0/1" if not bad_bin else f"*** FAIL: not 0/1 -> {bad_bin[:8]} ***"))

    # --- ITERATION 4: POST_GST_CUT must be 1 across the window ----------------
    if "POST_GST_CUT" in win.columns:
        print("POST_GST_CUT in window:", "1 on every day" if (win["POST_GST_CUT"] == 1).all()
              else "*** FAIL: not 1 on every day ***")

    # --- ITERATION 4: scaled shared continuous columns -------------------------
    for c in petrol_cols:
        if c in win.columns:
            tr = cal.loc[cal[time_col] <= TR_END, c]
            print(f"{c}: training z in [{tr.min():.2f}, {tr.max():.2f}] | forecast window z in "
                  f"[{win[c].min():.2f}, {win[c].max():.2f}] | {win[c].nunique()} distinct value(s) in the window")
            if win[c].max() > tr.max() + 0.5 or win[c].min() < tr.min() - 0.5:
                print("   *** forecast values lie outside the training range (extrapolation) ***")

    # --- the festive blocks --------------------------------------------------
    for name, block, expect in [("N block", N_BLOCK, 27), ("D block", D_BLOCK, 10)]:
        present = [c for c in block if c in win.columns]
        missing = [c for c in block if c not in win.columns]

        print(f"\n{name}: {len(present)}/{len(block)} columns present")
        if missing:
            print(f"  MISSING COLUMNS: {missing}")

        live = {c: int((win[c] != 0).sum()) for c in present}
        n_live = sum(1 for v in live.values() if v > 0)
        print(f"  columns with a non-zero day in the window: {n_live}/{expect}")

        if n_live == 0:
            print("  *** FAIL: the entire block is zero across the forecast window. ***")
            print("      The model is flying blind on this festival. This is the bug.")
        elif n_live < expect:
            dead = [c for c, v in live.items() if v == 0]
            print(f"  *** PARTIAL: these are all-zero: {dead}")
        else:
            print("  OK: every column fires at least once.")

        # each flag should fire on exactly one day
        multi = {c: v for c, v in live.items() if v > 1}
        if multi:
            print(f"  NOTE: fire on >1 day (expected 1 each): {multi}")

    # --- where does each anchor land -----------------------------------------
    for anchor in ["N", "D"]:
        if anchor in win.columns:
            hits = win.loc[win[anchor] != 0, time_col]
            print(f"\n'{anchor}' day-0 in window: "
                  f"{[d.date().isoformat() for d in hits] or 'NONE'}")

    # --- days with no festive signal at all ----------------------------------
    fest = [c for c in (N_BLOCK + D_BLOCK) if c in win.columns]
    if fest:
        blank = win.loc[(win[fest] == 0).all(axis=1), time_col]
        print(f"\nDays in the window with NO festive flag: {len(blank)}/{N_DAYS}")
        if len(blank):
            print(f"  first: {blank.min().date()}   last: {blank.max().date()}")
            by_month = blank.dt.month_name().value_counts()
            print("  by month: " + ", ".join(f"{m} {n}" for m, n in by_month.items()))
            print("  On these days the model has no festive signal and can only")
            print("  extrapolate the baseline from the encoder.")
    print()


# ------------------------------------------------------------------ ITERATION 4: spend paths
sp_path = os.path.join(CACHE_DIR, "spend_paths.npy")
if os.path.exists(sp_path):
    print("=" * 70)
    print("spend_paths.npy (per-series covariates)")
    print("=" * 70)
    sp = np.load(sp_path)
    days = pd.date_range(ROLES["train_start"], FC_END, freq=ROLES["freq"])
    print(f"Shape: {sp.shape} = (paths, days, columns); days run {days[0].date()} -> {days[-1].date()}")
    if sp.shape[1] != len(days):
        print(f"*** FAIL: expected {len(days)} days. ***")
    if not np.isfinite(sp).all():
        print("*** FAIL: NaN/inf in the spend paths. ***")
    i0, i_last = (FC_START - days[0]).days, (FC_START - days[0]).days - 1
    i_tr = (TR_END - days[0]).days + 1
    for j, c in enumerate(series_covariates):
        tr_all, fc_all = sp[:, :i_tr, j], sp[:, i0:, j]
        print(f"{c}: training z in [{tr_all.min():.2f}, {tr_all.max():.2f}] | forecast window z in "
              f"[{fc_all.min():.2f}, {fc_all.max():.2f}]")
        if fc_all.max() > tr_all.max() + 0.5 or fc_all.min() < tr_all.min() - 0.5:
            print("   *** some forecast values lie outside the training range (extrapolation) ***")
    flat = np.isclose(sp[:, i0:, :], sp[:, i_last:i_last + 1, :], atol=1e-4).all(axis=(1, 2))
    print(f"Spend paths whose whole forecast window equals the {days[i_last].date()} value: "
          f"{int(flat.sum()):,} of {len(sp):,}")
    if flat.mean() > 0.5:
        print("*** WARNING: most paths are flat -- the planned spend looks carried forward. ***")
    print()


# ------------------------------------------------------------------ compare
if len(sources) == 2:
    print("=" * 70)
    print("DO THE TWO ARTIFACTS AGREE?")
    print("=" * 70)
    a, b = list(sources.values())
    ca = set(a.columns) - {time_col}
    cb = set(b.columns) - {time_col}
    if ca != cb:
        print(f"*** Column sets differ. only in pkl: {sorted(ca-cb)[:8]} | "
              f"only in parquet: {sorted(cb-ca)[:8]} ***")
    else:
        wa = a[(a[time_col] >= FC_START) & (a[time_col] <= FC_END)].set_index(time_col).sort_index()
        wb = b[(b[time_col] >= FC_START) & (b[time_col] <= FC_END)].set_index(time_col).sort_index()
        cols = sorted(ca)
        same = np.allclose(wa[cols].to_numpy(float), wb[cols].to_numpy(float), equal_nan=True)
        print("Values identical in the forecast window:", same)
        if not same:
            print("*** The pickle inference uses differs from the source parquet.")
            print("    Rebuild shared_cov.pkl. ***")